In [7]:
import numpy as np
import time
import os

In [8]:
#use current number of experiment data files to track if experiment is done
data_path = 'C:\\Users\\SPUD1\\Documents\\qm_workspace\\saved_experiments'
entries = os.listdir(data_path)
file_count = sum(os.path.isfile(os.path.join(data_path, entry)) for entry in entries)
print(file_count)        

4872


In [9]:
#define hardware instances from qudi
laser_control = dlnsec_laser
laser_switch = laser_switch_ni
qm_switch = qm_switch_ni
tracker = scanning_optimize_logic

In [10]:
#load switch keys
laser_switch.available_states
las_sw_dict = laser_switch.available_states
las_sw_key =  list(las_sw_dict.keys())[0]

qm_switch.available_states
qm_sw_dict = qm_switch.available_states
qm_sw_key =  list(qm_sw_dict.keys())[0]

In [5]:
#code to run NV tracking:
num_track = 2
laser_switch.set_state(las_sw_key,'Low') #disconnect AWG to laser
laser_control.laser.set_mode('LAS') #activate laser internal mode
for i in range(num_track):
    tracker.start_optimize()
    while tracker.module_state() != 'idle':
        time.sleep(1)  # Check if the track is done
laser_switch.set_state(las_sw_key,'High') #connect AWG to laser
laser_control.laser.set_mode('EXT') #activate laser extenal mode

KeyboardInterrupt: 

In [10]:
#code to trigger QM experiment after tracking:
#remember to run qm script with trigger function in qm_workspace environment
num_track = 1
laser_switch.set_state(las_sw_key,'Low') #disconnect AWG to laser
laser_control.laser.set_mode('LAS') #activate laser internal mode
for i in range(num_track):
    tracker.start_optimize()
    while tracker.module_state() != 'idle':
        time.sleep(1)  # Check if the track is done
laser_switch.set_state(las_sw_key,'High') #connect AWG to laser
laser_control.laser.set_mode('EXT') #activate laser extenal mode
qm_switch.set_state(qm_sw_key,'High') #trigger QM script
time.sleep(5)
qm_switch.set_state(qm_sw_key,'Low') #reset trigger

In [17]:
#code run qm experiment loop with tracking in between:
#remember to run qm script with trigger function in qm_workspace environment!
entries = os.listdir(data_path)
file_count = sum(os.path.isfile(os.path.join(data_path, entry)) for entry in entries)
file_count_k = file_count
num_exp = 50
num_track = 2
for k in range(num_exp):
    qm_switch.set_state(qm_sw_key,'Low') #reset QM trigger
    laser_switch.set_state(las_sw_key,'Low') #disconnect AWG to laser
    laser_control.laser.set_mode('LAS') #activate laser internal mode
    time.sleep(10) #wait for internal mode to activate
    
    for i in range(num_track): #perform track
        print(f"tracking step #{i+1} in progress...")
        tracker.start_optimize()
        while tracker.module_state() != 'idle':
            time.sleep(1)  # Check if the track is done
    print(f"running iteration #{k+1}")
    laser_switch.set_state(las_sw_key,'High') #connect AWG to laser
    laser_control.laser.set_mode('EXT') #activate laser extenal mode
    qm_switch.set_state(qm_sw_key,'High') #trigger QM script
    
    while file_count_k == file_count:  # Check if the experiment is done
        time.sleep(1)  
        entries = os.listdir(data_path)
        file_count_k = sum(os.path.isfile(os.path.join(data_path, entry)) for entry in entries)
    file_count = file_count_k

    qm_switch.set_state(qm_sw_key,'Low') #reset qm trigger

print("experiment complete!")

tracking step #1 in progress...
tracking step #2 in progress...
running iteration #1
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #2
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #3
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #4
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #5
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #6
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #7
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #8
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #9
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #10
tracking step #1 in progress...
tracking step #2 in progress...
running iteration #11
tracking step #1 in progress...
tracking step #2 in progress...

In [12]:
laser_control.laser.set_mode('EXT') #activate laser extenal mode
qm_switch.set_state(qm_sw_key,'Low') #reset qm trigger